In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/icu_mortality_cohort_demo.csv")

modeling_df = pd.read_csv(DATA_PATH)

print(modeling_df.shape)
modeling_df.head()

(128, 12)


,subject_id,hadm_id,stay_id,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit,hospital_expire_flag
0,10023771,20044587,33177122,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
1,10005909,20199380,36496303,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0
2,10003400,20214994,32128372,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU),0
3,10008454,20291550,31959184,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU),0
4,10019385,20297618,39268883,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU),0


In [2]:
modeling_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   subject_id            128 non-null    int64
 1   hadm_id               128 non-null    int64
 2   stay_id               128 non-null    int64
 3   gender                128 non-null    str  
 4   anchor_age            128 non-null    int64
 5   admission_type        128 non-null    str  
 6   admission_location    128 non-null    str  
 7   insurance             128 non-null    str  
 8   marital_status        118 non-null    str  
 9   race                  128 non-null    str  
 10  first_careunit        128 non-null    str  
 11  hospital_expire_flag  128 non-null    int64
dtypes: int64(5), str(7)
memory usage: 12.1 KB


In [3]:
modeling_df.describe()

,subject_id,hadm_id,stay_id,anchor_age,hospital_expire_flag
count,1.280000e+02,1.280000e+02,1.280000e+02,128.000000,128.000000
mean,1.001866e+07,2.515074e+07,3.502284e+07,61.710938,0.117188
std,1.099626e+04,2.973111e+06,2.861454e+06,15.691490,0.322907
min,1.000003e+07,2.004459e+07,3.005745e+07,21.000000,0.000000
25%,1.000905e+07,2.265295e+07,3.243798e+07,52.000000,0.000000
50%,1.001900e+07,2.495555e+07,3.504597e+07,63.000000,0.000000
75%,1.002384e+07,2.798653e+07,3.727403e+07,72.000000,0.000000
max,1.004002e+07,2.997458e+07,3.988077e+07,91.000000,1.000000


In [4]:
modeling_df.isnull().sum()

subject_id               0
hadm_id                  0
stay_id                  0
gender                   0
anchor_age               0
admission_type           0
admission_location       0
insurance                0
marital_status          10
race                     0
first_careunit           0
hospital_expire_flag     0
dtype: int64

In [5]:
(modeling_df.isnull().mean()*100).round(2)

subject_id              0.00
hadm_id                 0.00
stay_id                 0.00
gender                  0.00
anchor_age              0.00
admission_type          0.00
admission_location      0.00
insurance               0.00
marital_status          7.81
race                    0.00
first_careunit          0.00
hospital_expire_flag    0.00
dtype: float64

In [7]:
import missingno as msno

msno.matrix(modeling_df)

ModuleNotFoundError: No module named 'missingno'

In [8]:
X = modeling_df.drop(
    columns=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "hospital_expire_flag"
    ]
)

y = modeling_df["hospital_expire_flag"]

In [9]:
X.head()

,gender,anchor_age,admission_type,admission_location,insurance,marital_status,race,first_careunit
0,M,70,ELECTIVE,PHYSICIAN REFERRAL,Medicare,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)
1,F,40,OBSERVATION ADMIT,EMERGENCY ROOM,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)
2,F,72,URGENT,TRANSFER FROM SKILLED NURSING FACILITY,Medicare,MARRIED,BLACK/AFRICAN AMERICAN,Medical/Surgical Intensive Care Unit (MICU/SICU)
3,F,26,EW EMER.,EMERGENCY ROOM,Other,SINGLE,WHITE,Trauma SICU (TSICU)
4,M,44,URGENT,TRANSFER FROM HOSPITAL,Other,MARRIED,WHITE,Cardiac Vascular Intensive Care Unit (CVICU)


In [10]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: hospital_expire_flag, dtype: int64

In [11]:
X.isnull().sum()

gender                 0
anchor_age             0
admission_type         0
admission_location     0
insurance              0
marital_status        10
race                   0
first_careunit         0
dtype: int64

In [12]:
categorical_columns = X.select_dtypes(include="object").columns

X[categorical_columns] = X[categorical_columns].fillna("Unknown")

C:\Users\yaooz\AppData\Local\Temp\ipykernel_4696\3745579602.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(include="object").columns


In [13]:
numeric_columns = X.select_dtypes(include=["int64","float64"]).columns

X[numeric_columns] = X[numeric_columns].fillna(
    X[numeric_columns].median()
)

In [14]:
X.isnull().sum()

gender                0
anchor_age            0
admission_type        0
admission_location    0
insurance             0
marital_status        0
race                  0
first_careunit        0
dtype: int64

In [15]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [16]:
X.head()

,anchor_age,gender_M,admission_type_ELECTIVE,admission_type_EW EMER.,admission_type_OBSERVATION ADMIT,admission_type_SURGICAL SAME DAY ADMISSION,admission_type_URGENT,admission_location_EMERGENCY ROOM,admission_location_INFORMATION NOT AVAILABLE,admission_location_PACU,...,race_WHITE - BRAZILIAN,race_WHITE - OTHER EUROPEAN,first_careunit_Coronary Care Unit (CCU),first_careunit_Medical Intensive Care Unit (MICU),first_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),first_careunit_Neuro Intermediate,first_careunit_Neuro Stepdown,first_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),first_careunit_Surgical Intensive Care Unit (SICU),first_careunit_Trauma SICU (TSICU)
0,70,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,40,False,False,False,True,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,72,False,False,False,False,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,26,False,False,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,True
4,44,True,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [17]:
print(X.shape)

(128, 41)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [19]:
print(X_train.shape)
print(X_test.shape)

(102, 41)
(26, 41)


In [20]:
print(y_train.value_counts())

hospital_expire_flag
0    90
1    12
Name: count, dtype: int64


In [21]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

hospital_expire_flag
0    0.882353
1    0.117647
Name: proportion, dtype: float64
hospital_expire_flag
0    0.884615
1    0.115385
Name: proportion, dtype: float64


In [22]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

In [23]:
X_train.to_csv(PROCESSED_DIR/"X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR/"X_test.csv", index=False)

y_train.to_csv(PROCESSED_DIR/"y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR/"y_test.csv", index=False)

## Day 3 Summary

- Removed identifier columns
- Examined missing values
- Filled missing data
- Applied one-hot encoding
- Split the data into training and testing sets
- Created machine-learning-ready datasets